In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import os
import sys
from datetime import datetime
import argparse
import intake
from easygems import healpix as egh
import healpy as hp

In [15]:
# SCREAM ETC track file
etc_dir = "/pscratch/sd/w/wcmca1/hackathon/etc_tracks/"
etc_csv_file = f"{etc_dir}screamv2_ne120_hp8.etc_stitched_nodes.txt"
etc_csv_file

'/pscratch/sd/w/wcmca1/hackathon/etc_tracks/screamv2_ne120_hp8.etc_stitched_nodes.txt'

In [3]:
cof_dir = "/pscratch/sd/w/wcmca1/hackathon/cof_masks/stats/"
cof_parquet_file = f"{cof_dir}scream_etc_overlap_tracking.parquet"
cof_parquet_file

'/pscratch/sd/w/wcmca1/hackathon/cof_masks/stats/scream_etc_overlap_tracking.parquet'

In [4]:
# # Test COF file (delete later)
# cof_dir_test = "/pscratch/sd/w/wcmca1/hackathon/cof_masks/test//stats//"
# cof_parquet_file_test = f"{cof_dir_test}scream_etc_overlap_tracking.parquet"
# cof_parquet_file_test

In [5]:
def parse_etc_track_file(file_path, unstructured_mesh=True):
    """
    Parse ETC track data from text file.
    
    Parameters:
    -----------
    file_path : str
        Path to ETC track file
    unstructured_mesh : bool
        Whether using HEALPix (True) or regular grid (False)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: storm_id, grid_id, lon, lat, year, month, day, hour, base_time
    """
    print(f"Parsing ETC track file: {file_path}")
    sys.stdout.flush()
    
    storm_data = []
    with open(file_path, 'r') as f:
        storm_id = 0
        for line in f:
            line = line.strip()
            if line.startswith("start"):
                # New storm
                storm_id += 1
                num_timesteps, year, month, day, hour = map(int, line.split()[1:])
            else:
                # Storm details
                cols = line.split()
                if unstructured_mesh:
                    storm_data.append({
                        "storm_id": storm_id,
                        "grid_id": int(cols[0]),
                        "lon": float(cols[1]),
                        "lat": float(cols[2]),
                        "year": int(cols[-4]),
                        "month": int(cols[-3]),
                        "day": int(cols[-2]),
                        "hour": int(cols[-1]),
                        "base_time": np.datetime64(f"{int(cols[-4]):04d}-{int(cols[-3]):02d}-{int(cols[-2]):02d}T{int(cols[-1]):02d}:00")
                    })
                else:
                    storm_data.append({
                        "storm_id": storm_id,
                        "lon_id": int(cols[0]),
                        "lat_id": int(cols[1]),
                        "lon": float(cols[2]),
                        "lat": float(cols[3]),
                        "year": int(cols[-4]),
                        "month": int(cols[-3]),
                        "day": int(cols[-2]),
                        "hour": int(cols[-1]),
                        "base_time": np.datetime64(f"{int(cols[-4]):04d}-{int(cols[-3]):02d}-{int(cols[-2]):02d}T{int(cols[-1]):02d}:00")
                    })
    
    df = pd.DataFrame(storm_data)
    print(f"Parsed {len(df)} storm points from {df['storm_id'].nunique()} unique storms")
    sys.stdout.flush()
    
    return df

In [6]:
etc_df = parse_etc_track_file(etc_csv_file, unstructured_mesh=True)
etc_df

Parsing ETC track file: /pscratch/sd/w/wcmca1/hackathon/etc_tracks/screamv2_ne120_hp8.etc_stitched_nodes.txt
Parsed 18704 storm points from 1329 unique storms
Parsed 18704 storm points from 1329 unique storms


,storm_id,grid_id,lon,lat,year,month,day,hour,base_time
0,1,225172,328.349301,51.062119,2019,8,1,0,2019-08-01 00:00:00
1,1,225144,330.365876,51.836503,2019,8,1,6,2019-08-01 06:00:00
2,1,227884,332.185954,52.994706,2019,8,1,12,2019-08-01 12:00:00
3,1,227986,332.396932,53.956885,2019,8,1,18,2019-08-01 18:00:00
4,1,249882,330.896761,55.873462,2019,8,2,0,2019-08-02 00:00:00
...,...,...,...,...,...,...,...,...,...
18699,1329,663199,237.147256,-59.867208,2020,8,30,18,2020-08-30 18:00:00
18700,1329,662752,243.120830,-62.508739,2020,8,31,0,2020-08-31 00:00:00
18701,1329,661227,249.306602,-64.761070,2020,8,31,6,2020-08-31 06:00:00
18702,1329,661044,255.000040,-67.376398,2020,8,31,12,2020-08-31 12:00:00


In [7]:
cof_df = pd.read_parquet(cof_parquet_file)
cof_df

,etc_track,time,overlap_flag,ar_tracks,mcs_tracks
0,1,2019-08-01 18:00:00,1,[],[182.0]
1,2,2019-08-01 18:00:00,0,[],[]
2,3,2019-08-01 18:00:00,0,[],[]
3,4,2019-08-01 18:00:00,0,[],[]
4,5,2019-08-01 18:00:00,1,[],[114.0]
...,...,...,...,...,...
18445,1314,2020-08-28 06:00:00,0,[],[]
18446,1316,2020-08-28 06:00:00,0,[],[]
18447,1317,2020-08-28 06:00:00,0,[],[]
18448,1318,2020-08-28 06:00:00,3,[546.0],"[55940.0, 55949.0, 55985.0, 56011.0]"


In [8]:
# cof_df_test = pd.read_parquet(cof_parquet_file_test)
# # cof_df_test
# cof_df_test.loc[cof_df_test['etc_track'] == 14].sort_values(by='time')

In [9]:
cof_df.loc[cof_df['etc_track'] == 17].sort_values(by='time')

,etc_track,time,overlap_flag,ar_tracks,mcs_tracks
125,17,2019-08-04 06:00:00,3,[3.0],"[496.0, 597.0]"
257,17,2019-08-04 12:00:00,3,[3.0],"[496.0, 597.0, 665.0]"
328,17,2019-08-05 06:00:00,3,[3.0],[748.0]
399,17,2019-08-05 12:00:00,3,[3.0],"[748.0, 820.0]"
244,17,2019-08-05 18:00:00,3,[3.0],"[748.0, 820.0]"
156,17,2019-08-06 00:00:00,3,[3.0],"[759.0, 820.0]"
483,17,2019-08-06 06:00:00,3,[3.0],"[759.0, 820.0]"
143,17,2019-08-06 12:00:00,3,[3.0],[759.0]
362,17,2019-08-06 18:00:00,3,[3.0],[759.0]
456,17,2019-08-07 00:00:00,3,[3.0],[759.0]


# Combine the two dataframes

We need to merge `etc_df` and `cof_df` where:
- `storm_id` (in etc_df) == `etc_track` (in cof_df)
- `base_time` (in etc_df) == `time` (in cof_df)

Since both dataframes have these as common keys, we can use `pd.merge()`.

In [10]:
# Check the column names and data types before merging
print("etc_df columns:", etc_df.columns.tolist())
print("etc_df dtypes:")
print(etc_df[['storm_id', 'base_time']].dtypes)
print("\ncof_df columns:", cof_df.columns.tolist())
print("cof_df dtypes:")
print(cof_df[['etc_track', 'time']].dtypes)

etc_df columns: ['storm_id', 'grid_id', 'lon', 'lat', 'year', 'month', 'day', 'hour', 'base_time']
etc_df dtypes:
storm_id              int64
base_time    datetime64[ns]
dtype: object

cof_df columns: ['etc_track', 'time', 'overlap_flag', 'ar_tracks', 'mcs_tracks']
cof_df dtypes:
etc_track             int64
time         datetime64[ns]
dtype: object


In [11]:
# Merge the two dataframes
# Use left merge to keep all ETC tracks, even if they don't have COF data
combined_df = etc_df.merge(
    cof_df,
    left_on=['storm_id', 'base_time'],
    right_on=['etc_track', 'time'],
    how='left'  # Use 'left' to keep all etc_df rows, 'inner' for only matching rows
)

print(f"etc_df shape: {etc_df.shape}")
print(f"cof_df shape: {cof_df.shape}")
print(f"combined_df shape: {combined_df.shape}")
combined_df

etc_df shape: (18704, 9)
cof_df shape: (18450, 5)
combined_df shape: (18704, 14)


,storm_id,grid_id,lon,lat,year,month,day,hour,base_time,etc_track,time,overlap_flag,ar_tracks,mcs_tracks
0,1,225172,328.349301,51.062119,2019,8,1,0,2019-08-01 00:00:00,1.0,2019-08-01 00:00:00,0.0,[],[]
1,1,225144,330.365876,51.836503,2019,8,1,6,2019-08-01 06:00:00,1.0,2019-08-01 06:00:00,1.0,[],[31.0]
2,1,227884,332.185954,52.994706,2019,8,1,12,2019-08-01 12:00:00,1.0,2019-08-01 12:00:00,1.0,[],[31.0]
3,1,227986,332.396932,53.956885,2019,8,1,18,2019-08-01 18:00:00,1.0,2019-08-01 18:00:00,1.0,[],[182.0]
4,1,249882,330.896761,55.873462,2019,8,2,0,2019-08-02 00:00:00,1.0,2019-08-02 00:00:00,1.0,[],[182.0]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18699,1329,663199,237.147256,-59.867208,2020,8,30,18,2020-08-30 18:00:00,1329.0,2020-08-30 18:00:00,3.0,[551.0],[56068.0]
18700,1329,662752,243.120830,-62.508739,2020,8,31,0,2020-08-31 00:00:00,1329.0,2020-08-31 00:00:00,3.0,[551.0],"[56068.0, 56326.0, 56351.0, 56392.0]"
18701,1329,661227,249.306602,-64.761070,2020,8,31,6,2020-08-31 06:00:00,1329.0,2020-08-31 06:00:00,0.0,[],[]
18702,1329,661044,255.000040,-67.376398,2020,8,31,12,2020-08-31 12:00:00,1329.0,2020-08-31 12:00:00,0.0,[],[]


In [12]:
# Test: Look at storm_id=2 in the combined dataframe
combined_df.loc[combined_df['storm_id'] == 11]

,storm_id,grid_id,lon,lat,year,month,day,hour,base_time,etc_track,time,overlap_flag,ar_tracks,mcs_tracks
201,11,96070,177.250059,56.637410,2019,8,2,6,2019-08-02 06:00:00,11.0,2019-08-02 06:00:00,3.0,[1.0],"[132.0, 233.0, 236.0]"
202,11,179879,181.415035,60.623664,2019,8,2,12,2019-08-02 12:00:00,11.0,2019-08-02 12:00:00,3.0,[1.0],"[132.0, 233.0, 236.0, 288.0]"
203,11,180118,184.062445,63.448577,2019,8,3,0,2019-08-03 00:00:00,11.0,2019-08-03 00:00:00,3.0,[1.0],"[132.0, 233.0, 421.0]"
204,11,191012,187.554695,64.761143,2019,8,3,6,2019-08-03 06:00:00,11.0,2019-08-03 06:00:00,3.0,[1.0],"[132.0, 233.0, 421.0, 474.0, 476.0]"
205,11,191235,190.439954,67.003576,2019,8,3,12,2019-08-03 12:00:00,11.0,2019-08-03 12:00:00,3.0,[1.0],"[474.0, 476.0, 510.0]"
206,11,190909,192.909794,67.562746,2019,8,3,18,2019-08-03 18:00:00,11.0,2019-08-03 18:00:00,3.0,[1.0],"[476.0, 510.0]"
207,11,190315,212.837822,69.608485,2019,8,4,0,2019-08-04 00:00:00,11.0,2019-08-04 00:00:00,2.0,[1.0],[]
208,11,193288,224.032257,72.942395,2019,8,4,6,2019-08-04 06:00:00,11.0,2019-08-04 06:00:00,2.0,[1.0],[]
209,11,193300,228.033712,73.681199,2019,8,4,12,2019-08-04 12:00:00,11.0,2019-08-04 12:00:00,0.0,[],[]
210,11,192994,230.500007,73.496566,2019,8,4,18,2019-08-04 18:00:00,11.0,2019-08-04 18:00:00,0.0,[],[]


In [13]:
# Optional: Clean up duplicate columns
# After merge, we have both 'storm_id' and 'etc_track' (which are the same)
# and both 'base_time' and 'time' (which are the same)
# We can drop the redundant columns

combined_clean_df = combined_df.drop(columns=['etc_track', 'time'])
print(f"Columns after cleanup: {combined_clean_df.columns.tolist()}")
combined_clean_df.loc[combined_clean_df['storm_id'] == 15]

Columns after cleanup: ['storm_id', 'grid_id', 'lon', 'lat', 'year', 'month', 'day', 'hour', 'base_time', 'overlap_flag', 'ar_tracks', 'mcs_tracks']


,storm_id,grid_id,lon,lat,year,month,day,hour,base_time,overlap_flag,ar_tracks,mcs_tracks
261,15,457516,173.671875,31.563258,2019,8,3,6,2019-08-03 06:00:00,3.0,[1.0],"[132.0, 233.0, 421.0, 474.0, 476.0]"
262,15,457664,174.375000,32.975000,2019,8,3,12,2019-08-03 12:00:00,3.0,[1.0],"[474.0, 476.0, 510.0]"
263,15,457693,175.253906,34.590737,2019,8,4,0,2019-08-04 00:00:00,2.0,[1.0],[]
264,15,87122,176.308594,38.301113,2019,8,4,12,2019-08-04 12:00:00,3.0,[1.0],[476.0]
265,15,87311,177.187500,40.033240,2019,8,4,18,2019-08-04 18:00:00,3.0,[1.0],[476.0]
266,15,87386,178.812308,41.810464,2019,8,5,0,2019-08-05 00:00:00,3.0,[1.0],"[476.0, 645.0, 702.0, 746.0]"
267,15,87421,179.820068,43.008737,2019,8,5,6,2019-08-05 06:00:00,3.0,[1.0],[772.0]
268,15,175011,180.933544,44.796120,2019,8,5,12,2019-08-05 12:00:00,3.0,[1.0],[772.0]
269,15,175055,181.738134,46.375204,2019,8,5,18,2019-08-05 18:00:00,3.0,[1.0],[772.0]
270,15,175059,182.510667,46.375201,2019,8,6,0,2019-08-06 00:00:00,3.0,[1.0],"[772.0, 849.0, 889.0, 919.0]"


In [17]:
out_parquet_file = f"{etc_dir}scream_etc_cof_data.parquet"

# Save combined DataFrame to parquet
combined_clean_df.to_parquet(out_parquet_file, index=True, engine='pyarrow')
print(f"Saved output to: {out_parquet_file}")

Saved output to: /pscratch/sd/w/wcmca1/hackathon/etc_tracks/scream_etc_cof_data.parquet


## Summary

The merge creates a combined dataframe where:
- Each row from `etc_df` is matched with corresponding rows from `cof_df` based on `storm_id`/`etc_track` and `base_time`/`time`
- Using `how='left'` keeps all ETC track points, even if there's no COF data (those will have NaN in COF columns)
- Using `how='inner'` would only keep rows where both datasets have matching entries
- After the merge, you can drop the redundant `etc_track` and `time` columns since they duplicate `storm_id` and `base_time`